# Chapitre 1.8 : l'aligner, les préférences et la DPO

Après l'affinage du chapitre 1.7, « ton IA » répond. Mais deux réponses à la même question ne se valent pas : l'une finit son tour, l'autre est coupée ; l'une respecte la consigne, l'autre l'oublie. Dans ce carnet, tu **classes deux réponses par des contrôles programmés**, sans juge, tu en tires une **paire** (une réponse choisie, une réponse rejetée), tu refais **à la main la perte DPO** qui apprend de ces paires, puis tu regardes ce qu'elle pousse : la réponse choisie vers le haut, ou la rejetée vers le bas.

Ce carnet accompagne [la page du chapitre 1.8](https://chipat-neko.github.io/forge-ia/partie-1/1-8/) : tu peux la lire avant, ou en même temps.

**Ce que tu vas faire**

1. noter deux réponses à la même invite, critère par critère, et voir laquelle gagne ;
2. écrire une fonction qui vérifie une consigne, comme le fait la forge ;
3. choisir une paire parmi plusieurs réponses, et voir pourquoi il n'y en a parfois pas ;
4. calculer la perte DPO à la main, et la comparer à celle de la forge ;
5. regarder quelques pas de DPO sur une paire, avec un modèle minuscule.

**Comment l'utiliser**

Lance les cellules de code de haut en bas : clique sur le bouton rond avec un triangle, à gauche de la cellule, ou tape Maj + Entrée. Tu peux aussi choisir « Exécution », puis « Tout exécuter ». Tu peux modifier le code et le relancer autant de fois que tu veux : tu ne peux rien casser.

Ce carnet calcule sur le processeur : le modèle de l'étape 4 est minuscule, et le calcul sur le processeur donne partout les mêmes nombres. Il ne télécharge aucune version de « ton IA ».

## Étape 0 : récupérer le code

Le code du projet est dans un dépôt GitHub public. La cellule suivante le télécharge avec `git clone`, installe la petite bibliothèque `regex` dont le tokenizer a besoin, puis indique à Python où trouver le projet et la forge.

In [ ]:
import os
import sys
from pathlib import Path

# Sur Colab, on télécharge le dépôt public. Quand ce carnet est lancé par les tests
# du projet, la variable FORGE_IA_LOCAL donne le chemin d’un dépôt déjà présent :
# on l’utilise, sans rien télécharger ni installer.
depot_local = os.environ.get("FORGE_IA_LOCAL")
if depot_local:
    depot = Path(depot_local)
else:
    depot = Path("forge-ia")
    if not depot.exists():
        !git clone --depth 1 https://github.com/chipat-neko/forge-ia
    !pip install --quiet regex

projet = depot / "projets" / "1.8-alignement"
if not projet.exists():
    raise FileNotFoundError(
        f"Le dossier {projet} est introuvable. Si le dépôt forge-ia était déjà là, "
        "supprime-le (panneau des fichiers, à gauche), puis relance cette cellule\u202f; sinon, vérifie ta connexion."
    )

# Python cherche les modules dans les dossiers de sys.path : on y ajoute ce projet, et forge.
sys.path.insert(0, str(projet.resolve()))
sys.path.insert(0, str((depot / "projets" / "forge").resolve()))

import json

import torch
from forge.conversation.format import Reponse, Tour
from forge.affinage.dpo import perte_dpo
from forge.entrainement.boucle import ReglagesDPO
from forge.evaluation.consignes import CONTRAINTES
from forge.preferences.invites import InvitePreference
from forge.preferences.noter import choisir_paire, noter, peut_etre_choisie, raison_de_la_paire, sans_paire

from alignement_pas_a_pas.noter import expliquer_note, note_de
from alignement_pas_a_pas.perte_dpo import derivee_a_la_main, perte_dpo_a_la_main, sigmoide, terme_sft_a_la_main
from alignement_pas_a_pas.poussee import TOKENIZER, lot_de_la_paire, modele_minuscule, paire_du_notebook, pousser
from forge.tokenizer import TokenizerOctets

appareil = "cpu"  # le modèle est minuscule : le processeur suffit, et donne partout les mêmes nombres
tokenizer = TokenizerOctets.charger(TOKENIZER)


def nombre(x, decimales=0):
    # À la française : une espace fine entre les milliers, une virgule pour les décimales.
    return f"{x:,.{decimales}f}".replace(",", "\u202f").replace(".", ",")


print(f"Ce carnet calcule sur le processeur, avec PyTorch {torch.__version__.split('+')[0]}.")
print(f"Le tokenizer de ton IA a {nombre(tokenizer.taille_vocabulaire)} jetons.")

## Étape 1 : deux réponses, et leur note

Pour classer deux réponses sans juge, la forge les **note** par cinq contrôles programmés, du plus important au moins important :

1. `terminee` : la réponse a-t-elle fini son tour, ou la limite de jetons l'a-t-elle coupée ?
2. `juste` : pour un problème de calcul, le dernier nombre du texte est-il la réponse attendue ? (Une question n'a rien à vérifier : le critère est acquis.)
3. `consignes` : combien des consignes de l'invite la réponse respecte-t-elle ?
4. `francais` : le texte est-il en français ? (Il n'est refusé que s'il est reconnu comme une autre langue.)
5. `longueur` : de 3 à 300 mots.

Deux notes se comparent dans cet ordre : le premier critère qui diffère décide. Une réponse coupée perd donc toujours, même si elle respecte les consignes. `expliquer_note` refait ce calcul critère par critère, et écrit pour chacun la raison en français. La cellule note deux réponses à une invite qui porte deux consignes, et vérifie que la note est celle de la forge (`noter`).

In [ ]:
# À toi : change l’invite, les consignes ou les réponses, puis relance la cellule.
question = "Quelle est la capitale de la France ? Réponds en 20 mots au plus. Écris toute ta réponse en minuscules."
consignes = (("mots_max", {"n": 20}), ("minuscules", {}))
invite = InvitePreference("carnet", (Tour("utilisateur", question),), consignes, None)

reponse_a = "paris est la capitale de la france."
reponse_b = "Paris est la capitale de la France, un pays d'Europe."


def afficher(nom, texte, terminee=True):
    explication = expliquer_note(texte, terminee, invite)
    print(f"Réponse {nom}\u202f: « {texte} »")
    for c in explication["criteres"]:
        print(f"  {'oui' if c['verdict'] else 'non'}  {c['nom']:<10} {c['valeur']}/{c['maximum']}  {c['raison']}")
    for c in explication["consignes"]:
        print(f"        {'respectée' if c['respectee'] else 'non respectée'}\u202f: {c['consigne']}")
    de_la_forge = noter(Reponse(texte=texte, pensee=None, ids=[], terminee=terminee), invite)
    print(f"  Même note que la forge\u202f: {note_de(explication) == de_la_forge}")
    return note_de(explication)


note_a = afficher("A", reponse_a)
print()
note_b = afficher("B", reponse_b)
print()
if note_a == note_b:
    print("Les deux notes sont égales\u202f: il n’y a pas de paire.")
else:
    gagnante, meilleure, pire = ("A", note_a, note_b) if note_a > note_b else ("B", note_b, note_a)
    print(f"{gagnante} gagne\u202f: le premier critère qui diffère est « {raison_de_la_paire(meilleure, pire)} ».")

## Exercice : vérifier une consigne

Les consignes de la forge sont des fonctions qui lisent le texte de la réponse et rendent vrai ou faux. L'une d'elles, `fin_imposee`, demande de **terminer la réponse par une phrase exacte**. Écris une fonction `finit_par(reponse, phrase)` qui fait la même vérification : vrai si la réponse, une fois ses blancs de fin retirés (espaces, retours à la ligne, espace insécable), se termine par `phrase`. Une phrase vide n'est jamais respectée.

La cellule compare ta fonction au vérificateur `fin_imposee` de la forge sur dix cas.

In [ ]:
fin_imposee = next(c for c in CONTRAINTES if c.id == "fin_imposee")


def finit_par(reponse, phrase):
    # À toi : renvoie True si la réponse se termine par la phrase. Remplace la ligne suivante.
    return False


cas = [
    ("Paris est belle.\nC’est tout pour aujourd’hui.", "C’est tout pour aujourd’hui."),
    ("C’est tout pour aujourd’hui.   \n", "C’est tout pour aujourd’hui."),
    ("C’est tout pour aujourd’hui. Merci.", "C’est tout pour aujourd’hui."),
    ("c’est tout pour aujourd’hui.", "C’est tout pour aujourd’hui."),
    ("Bonjour.", ""),
    ("", "Merci."),
    ("Merci.", "Merci."),
    ("Merci.\u00a0", "Merci."),
    ("Oui. Merci.\n\n", "Merci."),
    ("Merci beaucoup", "Merci."),
]
for reponse, phrase in cas:
    print(f"{reponse!r:<52} ta fonction {finit_par(reponse, phrase)!s:<5} la forge {fin_imposee.verifier(reponse, {'phrase': phrase})}")
if all(finit_par(reponse, phrase) == fin_imposee.verifier(reponse, {"phrase": phrase}) for reponse, phrase in cas):
    print("Défi réussi, bravo.")
else:
    print("Pas encore. Retire les blancs de fin de la réponse, puis regarde comment elle se termine.")

<details>
<summary>Voir une correction (essaie d’abord !)</summary>

`str.rstrip()` retire les blancs de fin d'un texte : les espaces, les retours à la ligne, mais aussi l'espace insécable. `str.endswith` dit ensuite si le texte se termine par la phrase. Une phrase vide termine n'importe quel texte : il faut la refuser à part.

Remplace la fonction de la cellule de l'exercice par :

```python
def finit_par(reponse, phrase):
    return phrase != "" and reponse.rstrip().endswith(phrase)
```

</details>

## Étape 2 : une paire parmi plusieurs réponses

Pour construire les paires de la DPO, la forge fait écrire **deux réponses** à « ton IA » pour chaque invite, les note, et garde la meilleure et la pire : c'est `choisir_paire`. Elle n'en garde pas toujours une :

- si toutes les notes sont égales, rien ne départage les réponses (`egalite`) ;
- si la meilleure est coupée, ou fausse pour un problème, elle ne peut pas être la réponse choisie : la DPO la prend pour modèle, et son terme SFT l'apprend mot à mot (`aucune_bonne_reponse`).

La cellule note quatre réponses à l'invite de l'étape 1, puis deux autres jeux de notes qui ne donnent pas de paire.

In [ ]:
reponses = [
    ("paris est la capitale de la france.", True),
    ("Paris est la capitale de la France.", True),
    ("paris est la capitale de la france, un pays d'europe.", True),
    ("paris est la capitale de la france et", False),
]
notes = [note_de(expliquer_note(texte, terminee, invite)) for texte, terminee in reponses]
for rang, ((texte, terminee), note) in enumerate(zip(reponses, notes)):
    print(f"  réponse {rang}\u202f: note {tuple(note)}{'' if terminee else ' (coupée)'}  « {texte} »")

meilleure, pire = choisir_paire(notes)
print(f"La paire\u202f: la choisie est la réponse {meilleure}, la rejetée la réponse {pire}. Premier critère qui diffère\u202f: « {raison_de_la_paire(notes[meilleure], notes[pire])} ».")
print(f"Elle est gardée\u202f: {sans_paire(notes) is None}.")

# Deux jeux de notes qui ne donnent pas de paire.
egales = [notes[0], notes[2]]
print(f"Deux réponses à égalité\u202f: choisir_paire donne {choisir_paire(egales)}, sans_paire donne « {sans_paire(egales)} ».")
coupees = [notes[3], note_de(expliquer_note("paris", False, invite))]
print(f"Deux réponses coupées\u202f: la meilleure peut être choisie\u202f? {peut_etre_choisie(max(coupees))}. sans_paire donne « {sans_paire(coupees)} ».")

## Étape 3 : la perte DPO à la main

Pour chaque paire, il faut quatre nombres : la log-probabilité de chaque réponse (la somme des log-probabilités de ses jetons) selon la **politique**, le modèle qu'on entraîne (`pc` pour la choisie, `pr` pour la rejetée), et selon la **référence**, la version de départ qui ne bouge pas (`qc` et `qr`). Le **gain** d'une réponse est l'écart entre la politique et la référence. La perte pousse le gain de la choisie au-dessus de celui de la rejetée :

```
z = β ((pc − qc) − (pr − qr))
perte = −ln σ(z)
```

où σ est la sigmoïde, et β la force de la préférence. La forge ajoute à cette perte un **terme SFT** : la perte par jeton de la réponse choisie, `−pc / lc`. Sans lui, la DPO peut écarter les deux réponses en les rendant toutes deux moins probables.

La cellule calcule la perte à la main, en nombres ordinaires, et la compare à `perte_dpo` de la forge, sur des tenseurs en float64.

In [ ]:
pc, pr = -12.0, -20.0  # log-probabilités de la politique : la choisie, la rejetée
qc, qr = -14.0, -18.0  # celles de la référence
beta = 0.1
lc = 10  # jetons appris de la choisie

gain_choisie, gain_rejetee = pc - qc, pr - qr
z = beta * (gain_choisie - gain_rejetee)
a_la_main = perte_dpo_a_la_main(pc, pr, qc, qr, beta)
sft = terme_sft_a_la_main(pc, lc)

print(f"Gain de la choisie\u202f: {nombre(gain_choisie, 2)}, gain de la rejetée\u202f: {nombre(gain_rejetee, 2)}.")
print(f"z = β × (écart des gains) = {nombre(z, 3)}, σ(z) = {nombre(sigmoide(z), 4)}.")
print(f"Perte DPO à la main\u202f: {nombre(a_la_main, 6)}.")
print(f"Terme SFT à la main\u202f: {nombre(sft, 6)} (−pc / lc, soit {nombre(-pc, 1)} sur {lc} jetons).")


def t(x):
    return torch.tensor([x], dtype=torch.float64)


de_forge, _ = perte_dpo(t(pc), t(pr), t(qc), t(qr), t(lc), t(lc), ReglagesDPO(beta=beta, poids_sft=0.0))
total, _ = perte_dpo(t(pc), t(pr), t(qc), t(qr), t(lc), t(lc), ReglagesDPO(beta=beta, poids_sft=1.0))
print(f"Perte DPO de la forge\u202f: {nombre(de_forge.item(), 6)}.")
print(f"Elles sont égales à 10⁻¹² près\u202f: {abs(a_la_main - de_forge.item()) <= 1e-12}")
print(f"Avec le terme SFT, la perte totale de la forge vaut {nombre(total.item(), 6)}\u202f: {abs(total.item() - (a_la_main + sft)) <= 1e-12} (DPO + SFT).")

# Ce que β change : la perte d’une paire dont la choisie gagne de plus en plus sur la rejetée.
print()
print("Perte selon l’écart des gains (choisie moins rejetée)\u202f:")
print(f"  {'écart':>6}" + "".join(f"  β = {nombre(b, 2):<6}" for b in (0.05, 0.1, 0.5)))
for ecart in (-4.0, 0.0, 4.0, 8.0, 16.0):
    print(f"  {nombre(ecart, 1):>6}" + "".join(f"  {nombre(perte_dpo_a_la_main(ecart, 0.0, 0.0, 0.0, b), 4):<10}" for b in (0.05, 0.1, 0.5)))
print(f"La dérivée selon la choisie vaut {nombre(derivee_a_la_main(beta, gain_choisie, gain_rejetee), 4)}\u202f: négative, monter la choisie fait baisser la perte.")

## Étape 4 : ce que la DPO pousse

La perte ne regarde que l'**écart** entre les deux gains : elle baisse autant si la choisie monte que si la rejetée descend. Rien ne dit lequel arrive. La cellule suivante le montre sur **une seule paire** et un modèle minuscule tiré au hasard : elle fait quelques pas de DPO avec le vrai code de la forge, et relève à chaque pas la log-probabilité moyenne par jeton de chaque réponse.

Elle le fait deux fois : avec la DPO seule, puis avec le terme SFT de l'étape 3. Les réglages (β, nombre de pas, taux) sont des variables au début de la cellule. Elle travaille sur la première vraie paire du dépôt quand `mesures/paires-exemples.json` existe ; sinon, elle le dit, et prend une paire fabriquée.

In [ ]:
paire, reelle = paire_du_notebook(projet / "mesures" / "paires-exemples.json")
if reelle:
    print("Ce carnet travaille sur la première vraie paire de mesures/paires-exemples.json.")
else:
    print("Aucune vraie paire n’est encore publiée\u202f: ce carnet travaille sur une paire fabriquée.")
print(f"Invite\u202f: « {paire['invite'][-1]['texte']} »")
print(f"Choisie\u202f: « {paire['choisie']['texte']} »")
print(f"Rejetée\u202f: « {paire['rejetee']['texte']} »{'' if paire['rejetee']['terminee'] else ' (coupée)'}")

lot = lot_de_la_paire(tokenizer, paire)
contexte = max(128, lot.ids.shape[1] + 1)
beta_essai, pas_essai, taux_essai = 0.1, 40, 1e-3  # les réglages de l’essai : change-les, puis relance
for poids_sft in (0.0, 1.0):
    modele = modele_minuscule(contexte)  # le même modèle de départ à chaque essai
    reglages = ReglagesDPO(beta=beta_essai, poids_sft=poids_sft)
    releves = pousser(modele, lot, reglages, pas=pas_essai, taux=taux_essai)
    config = modele.config
    print()
    print(f"{'DPO seule' if poids_sft == 0 else 'DPO et terme SFT'} (β {nombre(beta_essai, 2)}, {pas_essai} pas, taux {nombre(taux_essai, 4)}, "
          f"modèle de {config.couches} couches et de vecteurs de {config.dim} nombres)\u202f:")
    print(f"  {'pas':>4}  {'choisie':>9}  {'rejetée':>9}  {'marge':>7}  {'perte':>7}")
    for r in releves[::10]:
        print(f"  {r['pas']:>4}  {nombre(r['logps_choisie'], 3):>9}  {nombre(r['logps_rejetee'], 3):>9}  {nombre(r['marge'], 2):>7}  {nombre(r['perte'], 3):>7}")
    debut, fin = releves[0], releves[-1]
    monte = fin["logps_choisie"] - debut["logps_choisie"]
    baisse = debut["logps_rejetee"] - fin["logps_rejetee"]
    print(f"  La choisie a {'monté' if monte >= 0 else 'baissé'} de {nombre(abs(monte), 2)} par jeton, la rejetée a {'baissé' if baisse >= 0 else 'monté'} de {nombre(abs(baisse), 2)}.")
    print(f"  La marge a grandi\u202f: {'oui' if fin['marge'] > debut['marge'] else 'non'}. La perte a baissé\u202f: {'oui' if fin['perte'] < debut['perte'] else 'non'}. "
          f"La log-probabilité de la choisie a {'monté' if monte >= 0 else 'baissé'}.")

Lis les deux tableaux et les trois verdicts que le code vient de calculer, sans te fier à ce que tu attends : ils valent pour cette paire, ce modèle et ces réglages, et pas pour « ton IA ». Ce qui tient dans tous les cas : la perte ne regarde que l'écart entre les deux gains, donc rien ne force la choisie à monter ; le terme SFT ajoute une perte par jeton sur la choisie, qui tire sa log-probabilité vers le haut ; et si la choisie baisse malgré tout, c'est le piège connu de la DPO. La page du chapitre montre ce qui s'est vraiment passé sur « ton IA », avec les courbes de l'entraînement.

## Dernière étape : le même résultat que la page ?

La page du site n'utilise pas Python : elle refait deux calculs en TypeScript, directement dans ton navigateur, la note d'une réponse (les huit consignes, la langue et la longueur) et la perte DPO d'une paire. Pour être sûr que les deux donnent les mêmes résultats, le dépôt garde un fichier de référence, `tests/attendus.json`, calculé une fois pour toutes : quarante textes de référence, dont les pièges (accent combinant, apostrophe courbe, emoji, texte vide, dix mille caractères), et une grille de 27 points pour la perte. Les tests de la page relisent ce fichier. La cellule suivante le relit aussi, et le compare à ce que le code de ce carnet vient de calculer.

In [ ]:
attendus = json.loads((projet / "tests" / "attendus.json").read_text(encoding="utf-8"))
par_id = {c.id: c for c in CONTRAINTES}

# Chaque texte de référence donne-t-il les mêmes verdicts et la même note ?
textes_differents = []
for cas in attendus["textes"]:
    invite_du_cas = InvitePreference("reference", (), tuple((nom, kwargs) for nom, kwargs in cas["invite"]), None)
    explication = expliquer_note(cas["texte"], cas["terminee"], invite_du_cas)
    verdicts = {c["id"]: bool(par_id[c["id"]].verifier(cas["texte"], c["kwargs"])) for c in attendus["consignes"]}
    note = {nom: valeur for nom, valeur in explication["note"].items() if nom != "juste"}
    if (verdicts, note, explication["mots"], explication["langue"]) != (cas["verdicts"], cas["note"], cas["mots"], cas["langue"]):
        textes_differents.append(cas["nom"])

# Chaque point de la grille donne-t-il la même perte, à 10⁻¹² près ?
points_differents = []
for p in attendus["grille_dpo"]:
    perte = perte_dpo_a_la_main(p["ecart_choisie"], p["ecart_rejetee"], 0.0, 0.0, p["beta"])
    derivee = derivee_a_la_main(p["beta"], p["ecart_choisie"], p["ecart_rejetee"])
    if abs(perte - p["perte"]) > 1e-12 or abs(derivee - p["derivee"]) > 1e-12:
        points_differents.append((p["beta"], p["ecart_choisie"], p["ecart_rejetee"]))

print(f"{len(attendus['textes'])} textes et {len(attendus['grille_dpo'])} points de la grille comparés.")
for nom in textes_differents[:3]:
    print(f"Texte différent\u202f: {nom}.")
for point in points_differents[:3]:
    print(f"Point différent\u202f: β, écart de la choisie, écart de la rejetée = {point}.")

identique = not textes_differents and not points_differents
print(f"Même résultat que la page du site\u202f: {'oui' if identique else 'non'}")  # l’espace avant les deux-points est une espace fine insécable

In [ ]:
print("Ce carnet a tourné sur le processeur.")

## Pour aller plus loin

- Le code complet du projet, avec ses tests : [projets/1.8-alignement sur GitHub](https://github.com/chipat-neko/forge-ia/tree/main/projets/1.8-alignement).
- La page du chapitre : [chapitre 1.8](https://chipat-neko.github.io/forge-ia/partie-1/1-8/).
- Depuis le dossier du projet, `uv run --extra cpu python -m alignement_pas_a_pas noter "Ta réponse." --consigne minuscules` note une réponse critère par critère, `dpo` refait la perte d'une paire, et `pousser` refait l'étape 4.

Le code de ce carnet est sous licence MIT, ses textes sous licence CC BY 4.0.